In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# Initialize the Groq multimodal model
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

agent = create_agent(
    model=llm,)


In [9]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [11]:
print(uploader.value)

({'name': 'AHM_Champion.png', 'type': 'image/png', 'size': 247049, 'content': <memory at 0x0000020A20340640>, 'last_modified': datetime.datetime(2026, 5, 6, 11, 42, 47, 164000, tzinfo=datetime.timezone.utc)},)


In [12]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [17]:
from langchain.messages import HumanMessage
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The image presents a circular logo with a white background, featuring a prominent design that suggests it is related to a sports event or competition. The logo is divided into sections, each containing distinct elements.

*   **Outer Circle**
    *   The outer circle of the logo features the text "AGENTIC PREMIER LEAGUE" in gray letters, curved around the top half of the circle.
    *   At the bottom of the circle, the word "CHAMPION" is displayed in red letters.
*   **Inner Design**
    *   Inside the circle, a colorful graphic depicts a person wearing a blue shirt, red shorts, and green boots, engaged in a dynamic pose as if playing cricket.
    *   Above the graphic, the text "Google Cloud" appears in multicolored letters, matching the colors of the Google logo.
    *   Below the graphic, the city name "AHMEDABAD" is written in blue letters, accompanied by five yellow stars underneath.
*   **Background**
    *   The background of the image is plain white, providing a clean and neutr

In [18]:
## audio input

In [28]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")


Recording...


100%|██████████| 50/50 [00:05<00:00,  9.83it/s]

Done.


In [39]:
from google import genai
import base64
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

with open("audio.wav", "rb") as f:
    aud_b64 = f.read()

response = client.models.generate_content(
    model="gemini-2.5-pro",
    contents=[
        "Tell me about this audio.",
        {
            "mime_type": "audio/wav",
            "data":aud_b64 ,
        },
    ],
)

print(response.text)

FileNotFoundError: [Errno 2] No such file or directory: 'audio.wav'

In [ ]:
import os
import io
import time
import sounddevice as sd
from scipy.io.wavfile import write
from tqdm import tqdm

    from google import genai
from google.genai import types

# -------------------------
# Gemini API Key
# -------------------------
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# -------------------------
# Recording Settings
# -------------------------
duration = 5
sample_rate = 44100

print("🎤 Recording...")

audio = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype="int16"
)

for _ in tqdm(range(duration * 10)):
    time.sleep(0.1)

sd.wait()

print("✅ Recording Finished!")

# -------------------------
# Save audio in memory
# -------------------------
buffer = io.BytesIO()
write(buffer, sample_rate, audio)

audio_bytes = buffer.getvalue()

# -------------------------
# Send to Gemini
# -------------------------
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        "answer the audio questions",
        types.Part.from_bytes(
            data=audio_bytes,
            mime_type="audio/wav",
        ),
    ],
)

print("\nGemini Response:\n")
print(response.text)

🎤 Recording...


100%|██████████| 50/50 [00:05<00:00,  9.76it/s]


✅ Recording Finished!

Gemini Response:

The audio asks:
1.  "Can you tell me about the weather?"
2.  "How's the weather?"

To give you an accurate weather update, I need to know your current location or the specific location you're interested in. Once you provide that, I'd be happy to tell you how the weather is there!
